# 2025 DL Lab6: Text Summarization with Seq2Seq Model

Before we start, please put **your name** and **SID** in following format: <br>
Hi I'm 陸仁賈, 314831000.

**Your Answer:**    
Hi I'm 吳禎哲, 313833003.

## Overview
This assignment involves implementing a hybrid sequence-to-sequence model to perform text summarization on the SAMSum and Reddit TIFU datasets.

The model architecture is composed of two main parts:
A pre-trained model utilized as the encoder.
A new decoder which must be implemented from scratch.

The objective is to fine-tune the existing encoder while training the custom decoder from the beginning, enabling the complete model to generate accurate and concise summaries. Performance is measured using the standard summarization metric: ROUGE-L Score.

## Kaggle Competition
Kaggle is an online community of data scientists and machine learning practitioners. Kaggle allows users to find and publish datasets, explore and build models in a web-based data-science environment, work with other data scientists and machine learning engineers, and enter competitions to solve data science challenges.

This assignment use kaggle to calculate your grade.  
Please use this [**LINK**](https://www.kaggle.com/t/efb569a4c0774de681e9f8426cfac364) to join the competition.

## Unzip Data

Unzip dataset.zip

### SAMSum
+ `train` : 14700
+ `val` : 818
+ `test` : 819

### Redit_TIFU
+ `train` : 29498
+ `val` : 4212
+ `test` : 8429

In [1]:
# Auto-install wandb if missing (runs early under papermill)
import importlib, sys, subprocess
try:
    wandb = importlib.import_module('wandb')  # bind to name even if already installed
except ImportError:
    print('[INFO] wandb not found; installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'wandb'])
    import wandb
print('[INFO] wandb available:', wandb.__version__)

[INFO] wandb available: 0.23.0


In [2]:
import csv
import math
import random
from pathlib import Path
from typing import Optional, Tuple, Union
from data_utils import *
import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from transformer.Const import *
from transformer.Models import Seq2SeqModelWithFlashAttn
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODE = "train"  # set to "predict" for inference
CHECKPOINT_PATH = Path("checkpoints/latest.pt")
BEST_CHECKPOINT_PATH = Path("checkpoints/best.pt")
PREDICT_CHECKPOINT = Path("checkpoints/best.pt")
TIFU_TEST_PATH = Path("dataset/tifu/tifu_test.jsonl")
SAMSUN_TEST_PATH = Path("dataset/samsun/test.csv")
PREDICTION_OUTPUT = Path("result.csv")
MAX_GENERATION_LEN = MAX_TARGET_LEN
TRAIN_EPOCHS = 30
TRAIN_BATCH_SIZE = 100
GLOBAL_SEED = 42
NUM_WORKERS = 4
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## CREATE DATASET
use ConCate dataset to handle multiple datasets situation

In [4]:
def build_dataset(
    path: List[Optional[str]],
    tokenizer: PreTrainedTokenizerBase,
    require_target: bool = True,
) -> Optional[Dataset]:
    if all(p is None for p in path):
        return None
    datasets = []
    for p in path:
        if p is not None:
            dataset = SquadSeq2SeqDataset(
                Path(p), tokenizer, max_source_len=MAX_SOURCE_LEN, max_target_len=MAX_TARGET_LEN, require_target=require_target
            )
            datasets.append(dataset)
    print(f"Built dataset with {sum(len(ds) for ds in datasets)} samples.")
    if len(datasets) == 1:
        return datasets[0]
    return ConcatDataset(datasets)

def build_dataloader(
    source: Union[Optional[Dataset], Optional[str]],
    batch_size: int = 4,
    shuffle: bool = False,
    num_workers: int = 8,
) -> Optional[DataLoader]:
    dataset = source
    collator = QACollator # Don't forget to define QACollator in data_utils.py
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collator,
        num_workers=num_workers,
    )


## Main loop of your model

In [5]:
def run_epoch(
    dataloader: DataLoader,
    model: Seq2SeqModelWithFlashAttn,
    device: torch.device,
    optimizer: Optional[torch.optim.Optimizer],
    scheduler: Optional[object],
    pad_id: int,
    max_grad_norm: float,
    train: bool,
) -> float:
    model.train(train)
    total_loss = 0.0
    steps = 0
    iterator = tqdm(dataloader, desc="train" if train else "eval", leave=False)
    for batch in iterator:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)
        src_seq_len = batch["src_len"].to(device=device, dtype=torch.int32)
        tgt_seq_len = batch["tgt_len"].to(device=device, dtype=torch.int32)
        if torch.any(tgt_seq_len < 2):
            raise ValueError("Each target sequence must contain at least BOS and EOS tokens.")
        ############### YOUR CODE HERE ###############
        # Compute the loss
        # Hint: use model to get logits with teacher forcing, then compute loss with F.cross_entropy
        # Make sure to ignore the padding tokens in the loss computation
        ##############################################
        starts = torch.cumsum(tgt_seq_len, dim=0) - tgt_seq_len
        decoder_in_list = []
        labels_list = []
        new_tgt_seq_len_list = []
        for i, l in enumerate(tgt_seq_len.tolist()):
            seq = tgt[starts[i]: starts[i] + l]
            inp = seq[:-1]
            lab = seq[1:]
            decoder_in_list.append(inp)
            labels_list.append(lab)
            new_tgt_seq_len_list.append(l - 1)
        decoder_input_ids = torch.cat(decoder_in_list, dim=0)
        labels = torch.cat(labels_list, dim=0)
        decoder_seq_len = torch.tensor(new_tgt_seq_len_list, dtype=torch.int32, device=device)
        logits = model(src_input_ids=src, trg_input_ids=decoder_input_ids, src_seq_len=src_seq_len, trg_seq_len=decoder_seq_len)
        loss = F.cross_entropy(logits, labels, ignore_index=pad_id)
        if train:
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
        total_loss += loss.item()
        steps += 1
        iterator.set_postfix(loss=total_loss / max(1, steps))
    return total_loss / max(1, steps)


## Checkpoints management

In [6]:
def load_checkpoint(
    model: Seq2SeqModelWithFlashAttn,
    path: Path,
    device: torch.device,
) -> None:
    state = torch.load(path, map_location=device)
    model.load_state_dict(state["model_state_dict"])

def save_checkpoint(
    model: Seq2SeqModelWithFlashAttn,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[object],
    path: Path,
    epoch: int,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    if scheduler is not None and hasattr(scheduler, "state_dict"):
        state["scheduler_state_dict"] = scheduler.state_dict()
    torch.save(state, path)


## Training

In [7]:
### Hyperparameters and arguments ###
lr = 1e-4
weight_decay = 0.001
warmup_steps = 2000
epochs = TRAIN_EPOCHS
max_grad_norm = 1.0
batch_size = TRAIN_BATCH_SIZE
num_workers = NUM_WORKERS
#####################################
set_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    raise RuntimeError("CUDA is required to run this code.")

# Check if flash attention is available
try:
    import flash_attn  # noqa: F401
except ImportError:
    raise ImportError("flash_attn is required to run this code.")

# Weights & Biases init (graceful fallback if permission/API issues)
import os, wandb, json
wandb_run = None
_wandb_err = None
try:
    wandb_run = wandb.init(project=os.environ.get('WANDB_PROJECT','lab6-summarization'),
                           entity=os.environ.get('WANDB_ENTITY'),
                           config={
                               "lr": lr,
                               "weight_decay": weight_decay,
                               "warmup_steps": warmup_steps,
                               "epochs": epochs,
                               "max_grad_norm": max_grad_norm,
                               "batch_size": batch_size,
                               "num_workers": num_workers,
                               "model": "ModernBERT-base",
                           })
except Exception as e:
    _wandb_err = e
    print('[WARN] wandb.init failed, continue without remote logging:', e)
    # Optional: switch to offline if API key exists but perms denied
    if os.environ.get('WANDB_API_KEY') and not os.environ.get('WANDB_MODE'):
        os.environ['WANDB_MODE'] = 'offline'
        print('[INFO] Set WANDB_MODE=offline; run will save locally.')

model = Seq2SeqModelWithFlashAttn(
    transformer_model_path="answerdotai/ModernBERT-base",
    freeze_encoder=True,
).to(device)
if wandb_run is not None:
    wandb.watch(model, log="gradients", log_freq=100)
print(next(model.parameters()).device)
tokenizer = model.tokenizer
checkpoint_path = CHECKPOINT_PATH
best_checkpoint_path = BEST_CHECKPOINT_PATH
print('[INFO] wandb status:', 'active' if wandb_run else f'inactive ({_wandb_err})')

wandb: Currently logged in as: aaronwu901225main (NYCU_Deeplearning) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run gvz1rmtz


wandb: Tracking run with wandb version 0.23.0


wandb: Run data is saved locally in /home/at0842/aaronwu901225master.ai13/sundries/lab6/wandb/run-20251120_012000-gvz1rmtz
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run efficient-universe-1


wandb: ⭐️ View project at https://wandb.ai/NYCU_Deeplearning/lab6-summarization


wandb: 🚀 View run at https://wandb.ai/NYCU_Deeplearning/lab6-summarization/runs/gvz1rmtz


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


cuda:0
[INFO] wandb status: active


In [8]:
train_set = build_dataset(
    ["dataset/tifu/tifu_train.jsonl", "dataset/samsun/train.csv"],
    tokenizer=model.tokenizer,
)
train_loader = build_dataloader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
)
val_set = build_dataset(
    ["dataset/tifu/tifu_val.jsonl", "dataset/samsun/validation.csv"],
    tokenizer=model.tokenizer,
)
valid_loader = build_dataloader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=lr, weight_decay=weight_decay
)
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=min(warmup_steps, total_steps),
    num_training_steps=total_steps,
)

best_val_ppl = float("inf")
for epoch in range(1, epochs + 1):
    train_loss = run_epoch(
        train_loader,
        model,
        device,
        optimizer,
        scheduler,
        tokenizer.pad_token_id,
        max_grad_norm,
        train=True,
    )
    msg = f"Epoch {epoch}/{epochs} - train loss: {train_loss:.4f}"
    current_val_ppl = None
    with torch.no_grad():
        val_loss = run_epoch(
            valid_loader,
            model,
            device,
            optimizer=None,
            scheduler=None,
            pad_id=tokenizer.pad_token_id,
            max_grad_norm=max_grad_norm,
            train=False,
        )
    perplexity = math.exp(min(20, val_loss))
    current_val_ppl = perplexity
    msg += f" | val loss: {val_loss:.4f} | ppl: {perplexity:.2f}"
    print(msg)
    # wandb epoch logs
    try:
        import wandb
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_perplexity": perplexity,
            "lr": scheduler.get_last_lr()[0] if scheduler is not None else lr,
        })
    except Exception:
        pass
    if checkpoint_path is not None:
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            path=checkpoint_path,
            epoch=epoch,
        )
    if (
        current_val_ppl is not None
        and current_val_ppl < best_val_ppl
        and best_checkpoint_path is not None
    ):
        best_val_ppl = current_val_ppl
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            path=best_checkpoint_path,
            epoch=epoch,
        )

Built dataset with 44229 samples.


Built dataset with 5030 samples.


train:   0%|          | 0/443 [00:00<?, ?it/s]

eval:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 1/30 - train loss: 8.7654 | val loss: 6.9357 | ppl: 1028.30


...

eval:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 18/30 - train loss: 4.3726 | val loss: 4.5404 | ppl: 93.73


train:   0%|          | 0/443 [00:00<?, ?it/s]

eval:   0%|          | 0/51 [00:00<?, ?it/s]

Epoch 19/30 - train loss: 4.3600 | val loss: 4.5340 | ppl: 93.13


## Predict Result

Predict the labesl based on testing set. Upload to [Kaggle](https://www.kaggle.com/t/efb569a4c0774de681e9f8426cfac364).

**How to upload**

1. To kaggle. Click "Submit Predictions"
2. Upload the result.csv
3. System will automaticlaly calculate the accuracy of 50% dataset and publish this result to leaderboard.

In [ ]:
load_checkpoint(model, PREDICT_CHECKPOINT, device)
model.eval()
test_set = build_dataset(
    [TIFU_TEST_PATH, SAMSUN_TEST_PATH],
    tokenizer=model.tokenizer,
    require_target=False,
)
test_loader = build_dataloader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)
predictions: List[Tuple[str, str]] = []
with torch.no_grad():
    for sample in tqdm(test_loader, desc="predict", leave=False):
        input_ids = sample["src"].to(device)
        src_lens = sample["src_len"].to(device=device, dtype=torch.int32)
        ids = sample["id"] #list of ids
        summaries = model.generate(
            input_ids=input_ids,
            src_seq_len=src_lens,
            generation_limit=MAX_GENERATION_LEN,
            sampling=True,
            top_k=50,
            top_p=0.9,
        )
        predictions.extend(zip(ids, summaries))
output_path = PREDICTION_OUTPUT
write_predictions_csv(output_path, predictions)
print(f"Wrote {len(predictions)} predictions to {output_path}")